<figure>
<center>
<img src='https://www.economicas.uba.ar/wp-content/uploads/2020/08/cropped-logo_FCE.png' />
</figure>

# Polygon.io — API Financiera con Python

**Polygon.io** es una plataforma de datos financieros profesionales que provee precios históricos
e intradía de acciones, ETFs, criptos, noticias, etc.

A diferencia de `dolarapi.com` o `argentinadatos.com`, Polygon introduce dos conceptos nuevos:

| Concepto | Qué significa |
|----------|---------------|
| **Autenticación** | Hay que identificarse con una API key en cada request |
| **Paginación** | Cuando hay muchos resultados, la API los devuelve en páginas |

Esta notebook tiene dos partes:

1. **Consultas directas con `requests`** — construimos cada request a mano, manejamos la auth, los parámetros y la paginación
2. **Librería oficial `polygon-api-client`** — el mismo resultado en pocas líneas, y entendemos qué nos abstrae

---

### Obtener una API key gratuita

1. Ir a [polygon.io](https://polygon.io) → **Get your free API Key**
2. Crear cuenta (email + contraseña, sin tarjeta de crédito)
3. Obtener la key

**Free tier**: datos históricos ilimitados (diario e intradía), 5 requests por minuto.

---
### Parte 1 — Consultas directas con `requests`

Vamos a consumir la API de Polygon construyendo los requests HTTP a mano.
Así se ve exactamente qué viaja por la red y qué devuelve el servidor.

In [1]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import time

Configuración de Polygon

In [2]:
# Reemplazar con tu API key de polygon.io
API_KEY = "COMPLETAR ACA!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!"

BASE_URL = "https://api.polygon.io"

#### 1.1 Autenticación — cómo se envía la API key

Las APIs protegidas requieren que te identifiques en cada request.
Polygon soporta dos métodos equivalentes:

**Opción A — Query parameter** (la key viaja en la URL):
```
GET /v2/aggs/ticker/AAPL/range/1/day/2024-01-01/2024-12-31?apiKey=tu_key
```

**Opción B — Header de autorización** (más seguro, la key no queda en logs de URL):
```
GET /v2/aggs/ticker/AAPL/range/1/day/2024-01-01/2024-12-31
Authorization: Bearer tu_key
```

Vamos a usar el header, que es la práctica recomendada.

**Autenticación con Bearer Token en APIs**

Un **Bearer Token** (token de portador) es el estándar de seguridad utilizado por la mayoría de las API web, como Polygon, para verificar tu identidad y otorgarte acceso a los datos. 
Su nombre proviene de su premisa principal: **el portador de este token es quien recibe la autorización**.

En la práctica, funciona como una credencial o una llave digital que se adjunta en el encabezado (*header*) de cada petición HTTP. <br>
Al recibir una solicitud con la estructura `"Authorization": "Bearer <tu_api_key>"`, el servidor valida la clave y determina si tienes permisos para consultar la información solicitada.

In [ ]:
# Construimos el header de autenticación
headers = {
    "Authorization": f"Bearer {API_KEY}"
}

# Endpoint: /v2/aggs/ticker/{ticker}/range/{multiplier}/{timespan}/{from}/{to}
ticker="AAPL"
fecha_desde="2024-01-01"
fecha_hasta="2024-12-31"
url = f"{BASE_URL}/v2/aggs/ticker/{ticker}/range/1/day/{fecha_desde}/{fecha_hasta}"

# Query parameters opcionales
params = {
    "adjusted": "true",   # precios ajustados por dividendos y splits
    "sort":     "asc",    # orden cronológico
    "limit":    50000,    # máximo de resultados por página
}

resp = requests.get(url, headers=headers, params=params, timeout=10)
resp.raise_for_status()

# Inspeccionamos la URL final que se construyó
print(f"URL del request: {resp.url[:80]}...")
print(f"Status:          {resp.status_code}")

URL del request: https://api.polygon.io/v2/aggs/ticker/AAPL/range/1/day/2024-01-01/2024-12-31?adj...
Status:          200


#### 1.2 Estructura del JSON de respuesta

Polygon no devuelve una lista directa de resultados.
El JSON tiene una **estructura anidada** con metadatos y resultados separados:

In [4]:
data = resp.json()

# Metadatos del response
print(f"status:       {data['status']}")
print(f"resultsCount: {data.get('resultsCount')}")
print(f"count:        {data.get('count')}")
print(f"next_url:     {data.get('next_url', '(no hay — todos los resultados entraron en una página)')}")
print()

# Los datos OHLCV están dentro de la clave "results"
print("Primera barra:")
data['results'][0]

status:       OK
resultsCount: 59
count:        59
next_url:     (no hay — todos los resultados entraron en una página)

Primera barra:


{'v': 31855693.0,
 'vw': 225.0548,
 'o': 224.3,
 'c': 225.77,
 'h': 225.98,
 'l': 223.25,
 't': 1728360000000,
 'n': 495518}

La respuesta tiene esta forma:

```json
{
  "status":       "OK",
  "resultsCount": 250,
  "count":        250,
  "next_url":     "https://api.polygon.io/v2/aggs/...",   ← si hay más páginas
  "results": [
    { "t": 1704153600000, "o": 185.12, "h": 186.90, "l": 183.43, "c": 185.92, "v": 72309500, "vw": 185.07 },
    ...
  ]
}
```

Los campos de cada barra usan nombres cortos: `t` (timestamp en ms), `o`, `h`, `l`, `c`, `v`, `vw` (VWAP).

#### 1.3 Paginación

Cuando hay más resultados de los que entran en una respuesta, Polygon devuelve `next_url` — una URL lista para pedir la siguiente página.

```
Página 1 → results[0..999]  + next_url → "https://api.polygon.io/v2/aggs/...?cursor=abc123"
Página 2 → results[1000..1999] + next_url → ...
Página N → results[...]  (sin next_url → fin)
```

Para datos intradía (miles de barras), es necesario iterar sobre las páginas.

In [5]:
def paginar(url, params=None):
    """
    Recorre todas las páginas de un endpoint de Polygon que usa next_url.

    Parámetros
    ----------
    url    : str  — URL del primer request
    params : dict — query parameters del primer request (opcional)

    Retorna
    -------
    list — todos los resultados concatenados de todas las páginas
    """
    todos = []
    pagina = 1

    while url:
        resp = requests.get(url, headers=headers, params=params, timeout=10)
        resp.raise_for_status()
        data = resp.json()

        resultados = data.get('results', [])
        todos.extend(resultados)
        print(f"  Página {pagina}: {len(resultados)} resultados  (acumulado: {len(todos)})")

        # next_url ya incluye el cursor — no se repiten los params
        url    = data.get('next_url')
        params = None
        pagina += 1

    return todos


def get_aggs(ticker, multiplier, timespan, desde, hasta):
    """
    Descarga barras OHLCV de Polygon manejando paginación automáticamente.

    Parámetros
    ----------
    ticker     : str — símbolo bursátil (ej. 'AAPL')
    multiplier : int — cantidad de unidades por barra (ej. 1, 5, 15)
    timespan   : str — unidad de tiempo: 'minute', 'hour', 'day', 'week'
    desde      : str — fecha inicio 'YYYY-MM-DD'
    hasta      : str — fecha fin    'YYYY-MM-DD'

    Retorna
    -------
    pd.DataFrame con columnas: apertura, maximo, minimo, cierre, volumen, vwap
    """
    url = f"{BASE_URL}/v2/aggs/ticker/{ticker}/range/{multiplier}/{timespan}/{desde}/{hasta}"
    params = {"adjusted": "true", "sort": "asc", "limit": 50000}

    resultados = paginar(url, params)

    df = pd.DataFrame([{
        'fecha':    pd.to_datetime(r['t'], unit='ms', utc=True).tz_convert('America/New_York'),
        'apertura': r['o'],
        'maximo':   r['h'],
        'minimo':   r['l'],
        'cierre':   r['c'],
        'volumen':  r['v'],
        'vwap':     r.get('vw'),
    } for r in resultados])

    return df.set_index('fecha').sort_index()


print("Descargando AAPL diario 2024...")
df_aapl = get_aggs('AAPL', 1, 'day', '2024-01-01', '2024-12-31')
print(f"\nShape final: {df_aapl.shape}")
df_aapl.head()

Descargando AAPL diario 2024...
  Página 1: 59 resultados  (acumulado: 59)

Shape final: (59, 6)


,apertura,maximo,minimo,cierre,volumen,vwap
fecha,,,,,,
2024-10-08 00:00:00-04:00,224.30,225.98,223.25,225.77,31855693.0,225.0548
2024-10-09 00:00:00-04:00,225.23,229.75,224.83,229.54,33591091.0,228.3401
2024-10-10 00:00:00-04:00,227.78,229.50,227.17,229.04,28183544.0,228.5592
2024-10-11 00:00:00-04:00,229.30,229.41,227.34,227.55,31759188.0,228.0176
2024-10-14 00:00:00-04:00,228.70,231.73,228.60,231.30,39882085.0,230.7772


### 1.3.1 Datos intradía — Ejemplo de paginación

Con barras de 1 minuto para una semana completa la respuesta no entra en una sola página.
Acá es donde el `while url` del loop de paginación cobra sentido.

In [ ]:
# Barras de 1 minuto — una semana completa (~2.000 barras)
print("Descargando AAPL intradía (1 min), semana 4–8 nov 2024...")
df_min = get_aggs('AAPL', 1, 'minute', '2024-11-04', '2024-11-08')
print(f"\nShape: {df_min.shape}")
df_min.head()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True,
                          gridspec_kw={'height_ratios': [3, 1]})

# Precio de cierre
axes[0].plot(df_aapl.index, df_aapl['cierre'],
             color='steelblue', linewidth=1.5, label='Cierre')
axes[0].plot(df_aapl.index, df_aapl['vwap'],
             color='#c45c2a', linewidth=1, linestyle='--',
             alpha=0.7, label='VWAP')
axes[0].fill_between(df_aapl.index, df_aapl['minimo'], df_aapl['maximo'],
                      alpha=0.07, color='steelblue', label='Rango diario')
axes[0].set_title('Apple (AAPL) — Precio diario 2024', fontsize=13)
axes[0].set_ylabel('Precio (USD)')
axes[0].legend(fontsize=9)
axes[0].grid(axis='y', alpha=0.3)

# Volumen
axes[1].bar(df_aapl.index, df_aapl['volumen'] / 1e6,
            color='steelblue', alpha=0.5, width=0.8)
axes[1].set_ylabel('Volumen (M)')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
axes[1].grid(axis='y', alpha=0.3)

plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### 1.5 Otros endpoints — detalles de empresa y noticias

Los endpoints de Polygon tienen una estructura consistente:
siempre autenticación por header, siempre JSON con `results` adentro.

In [ ]:
# Detalles de empresa — /v3/reference/tickers/{ticker}
resp = requests.get(
    f"{BASE_URL}/v3/reference/tickers/AAPL",
    headers=headers, timeout=10
)
resp.raise_for_status()

# Los detalles vienen dentro de "results" (un dict, no una lista)
detalle = resp.json()['results']

campos = {
    'name':             'Nombre',
    'market':           'Mercado',
    'primary_exchange': 'Exchange',
    'sic_description':  'Sector',
    'total_employees':  'Empleados',
    'market_cap':       'Market Cap',
    'homepage_url':     'Sitio web',
}

for campo, etiqueta in campos.items():
    valor = detalle.get(campo, 'N/A')
    if campo == 'market_cap' and isinstance(valor, (int, float)):
        valor = f'USD {valor / 1e12:.2f} T'
    elif campo == 'total_employees' and isinstance(valor, int):
        valor = f'{valor:,}'
    print(f"  {etiqueta:<16} {valor}")

In [ ]:
# Noticias — /v2/reference/news
resp = requests.get(
    f"{BASE_URL}/v2/reference/news",
    headers=headers,
    params={"ticker": "AAPL", "limit": 5, "order": "desc"},
    timeout=10
)
resp.raise_for_status()

noticias = resp.json()['results']

print(f"Últimas {len(noticias)} noticias sobre AAPL:\n")
for i, n in enumerate(noticias, 1):
    fecha = pd.to_datetime(n['published_utc']).strftime('%d/%m/%Y %H:%M')
    print(f"{i}. [{fecha}]  {n['title']}")
    print(f"   Fuente: {n['publisher']['name']}")
    print()

---
### Parte 2 — Librería oficial `polygon-api-client`

Ahora hacemos exactamente lo mismo usando la librería oficial.
Compará la cantidad de código necesario.

```
pip install polygon-api-client
```

In [2]:

from polygon import RESTClient

client = RESTClient(api_key="API_KEY")
print("Cliente inicializado.")

Cliente inicializado.


### 2.1 Precio histórico — comparación directa

**Con `requests` lo que hicimos en Parte 1.**

**Con la librería:**

In [ ]:
# La librería maneja autenticación, paginación y deserialización automáticamente
aggs = client.get_aggs(
    ticker='AAPL', multiplier=1, timespan='day',
    from_='2024-01-01', to='2024-12-31', adjusted=True
)

# Devuelve objetos Python, no dicts — accedemos por atributo
df_lib = pd.DataFrame([{
    'fecha':    pd.to_datetime(a.timestamp, unit='ms'),
    'apertura': a.open,
    'cierre':   a.close,
    'volumen':  a.volume,
    'vwap':     a.vwap,
} for a in aggs]).set_index('fecha')

print(f"Shape: {df_lib.shape}")
df_lib.head()